# Fraud Detection Assignment

This focuses on understanding the dataset and performing 
data exploration before building fraud detection models.

## Dataset Design 

Instead of randomly marking transactions as fraud, the dataset was created
to show the realistic fraud behaviour that is seen in real payment systems.

## Design of the Fraud patterns:

Fraud is not obvious or easy to separate

Multiple features need to be considered together

Some normal transactions may look suspicious and some fraud may look normal

This makes the problem more realistic and allows a fair comparison between
different anomaly detection models, not just based on accuracy.

### Problem Constraints Assumed

False positives are more costly than false negatives as
flagging genuine users as fraud would disappoint the legit users. 
So we should minimize false alarms.

Fraud cases are rare (<2%).
I have created the synthetic dataset in a way that 
it looks like real transaction data.

Low prediction time is required (<100 ms).
Fraud detection must be fast enough to support real-time transaction
decisions.


In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

num_transactions = 100000

data = pd.DataFrame({
    "transaction_id": [f"TXN_{i:08d}" for i in range(num_transactions)],
    "customer_id": np.random.randint(1, 5001, num_transactions),
    "amount": np.random.exponential(scale=2500, size=num_transactions),
    "hour": np.random.randint(0, 24, num_transactions),
    "day_of_week": np.random.randint(0, 7, num_transactions),
    "distance_from_home": np.random.exponential(scale=8, size=num_transactions),
    "merchant_category": np.random.choice(
        ["grocery", "electronics", "restaurant", "fuel", "luxury"],
        size=num_transactions,
        p=[0.35, 0.2, 0.25, 0.15, 0.05]
    )
})

In [4]:
fraud_probability = (
    (data["amount"] > 15000).astype(int) * 0.35 +
    (data["distance_from_home"] > 25).astype(int) * 0.30 +
    (data["hour"].isin([0, 1, 2, 3])).astype(int) * 0.20 +
    (data["merchant_category"] == "luxury").astype(int) * 0.40
)

noise = np.random.uniform(0, 0.15, num_transactions)

data["fraud_score"] = fraud_probability + noise
data["is_fraud"] = (data["fraud_score"] > 0.6).astype(int)

data.drop(columns=["fraud_score"], inplace=True)


## Fraud Pattern Design Rationale

Fraud labels were not assigned randomly. I have choosen multiple weak features
to understand fraud behaviour:

Fraud transaction usually have high amounts. So the chance is high that the transaction could be fraud
Transaction that happens far from usual location is suspicious
Fraud happens at odd times compared to normal users
Cetrain categories like Luxury have high chances of fraud risk

I also added noise so that fraud is probabilistic by preventing rule based detection and also to avoid the model to cheat.


In [5]:
data.to_csv("../data/transactions.csv", index=False)
print("Updated dataset with intentional fraud patterns saved")

Updated dataset with intentional fraud patterns saved


In [6]:
df = pd.read_csv("../data/transactions.csv")
df.head()

,transaction_id,customer_id,amount,hour,day_of_week,distance_from_home,merchant_category,is_fraud
0,TXN_00000000,861,321.657107,12,6,8.349037,restaurant,0
1,TXN_00000001,3773,1944.835714,18,2,4.499823,grocery,0
2,TXN_00000002,3093,14551.485133,11,2,18.237703,grocery,0
3,TXN_00000003,467,977.794274,6,1,2.069479,restaurant,0
4,TXN_00000004,4427,106.712706,20,1,13.467285,restaurant,0


In [7]:
df["is_fraud"].value_counts(normalize=True)

is_fraud
0    0.98743
1    0.01257
Name: proportion, dtype: float64

### Dataset Specification 

The assignment mentions an approximate fraud rate of 2%.
In my synthetic dataset, the observed fraud rate is around 1.26%.